In [ ]:
#DATASET DIAGNOSTIC
# =====================================================================

import os
import cv2
import numpy as np
from pathlib import Path
from PIL import Image
import shutil

# ============================================
# CONFIGURATION - UPDATE THIS IF NEEDED
# ============================================
DATASET_PATH = "/kaggle/input/license_plate"
WORK_DIR = "/kaggle/working"

print("="*60)
print("LICENSE PLATE DATASET DIAGNOSTIC TOOL")
print("="*60)

# ============================================
# FIND ALL IMAGES
# ============================================
def find_all_images(path):
    """Find all image files in dataset"""
    extensions = ['.jpg', '.jpeg', '.png', '.bmp', '.JPG', '.JPEG', '.PNG']
    images = []
    for ext in extensions:
        images.extend(Path(path).rglob(f'**/*{ext}'))
    return images

print(f"\n🔍 Scanning dataset: {DATASET_PATH}")
all_images = find_all_images(DATASET_PATH)
print(f"✓ Found {len(all_images)} images")

# ============================================
# TEST IMAGE LOADING
# ============================================
def test_image_loading(img_path):
    """Test if image can be loaded with OpenCV and PIL"""
    errors = []
    try:
        img = cv2.imread(str(img_path))
        if img is None:
            errors.append("OpenCV: imread returned None")
    except Exception as e:
        errors.append(f"OpenCV: {str(e)}")
    try:
        img = Image.open(img_path)
        img.verify()
    except Exception as e:
        errors.append(f"PIL: {str(e)}")
    return errors

print("\n🧪 Testing image loading...")
problematic_images = []

for i, img_path in enumerate(all_images):
    if i % 100 == 0:
        print(f"   Tested {i}/{len(all_images)} images...", end='\r')
    errors = test_image_loading(img_path)
    if errors:
        problematic_images.append({'path': img_path, 'errors': errors})

print(f"   Tested {len(all_images)}/{len(all_images)} images    ")

# ============================================
# REPORT RESULTS
# ============================================
print(f"\n📊 DIAGNOSTIC RESULTS:")
print("="*60)

if not problematic_images:
    print("✅ All images are OK!")
else:
    print(f"⚠️  Found {len(problematic_images)} problematic images")

print("="*60)



In [ ]:
print("📦 Installing packages...")
!pip install -q ultralytics opencv-python-headless PyYAML
print("✓ Done\n")

import os
import yaml
from pathlib import Path
import matplotlib.pyplot as plt
import cv2
import numpy as np
from ultralytics import YOLO
from IPython.display import display, Image as IPImage
import torch
import shutil

print(f"✓ GPU: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   {torch.cuda.get_device_name(0)}\n")


In [ ]:
datasets = os.listdir('/kaggle/input')
print(f"Available datasets: {datasets}")

# Find the license plate dataset
DATASET = "/kaggle/input/license_plate"

# Verify dataset exists
if not os.path.exists(DATASET):
    for d in datasets:
        if 'license' in d.lower() or 'plate' in d.lower():
            DATASET = f"/kaggle/input/{d}"
            break
    print(f"Found dataset at: {DATASET}")

# Output directory
OUTPUT_DIR = "/kaggle/working/outputs"
MODEL_DIR = f"{OUTPUT_DIR}/model"
os.makedirs(MODEL_DIR, exist_ok=True)

print("="*60)
print("CONFIGURATION")
print("="*60)
print(f"Dataset: {DATASET}")
print(f"Output Dir: {OUTPUT_DIR}")
print(f"Model Dir: {MODEL_DIR}")

# Training config
EPOCHS = 100
BATCH = 16
IMG_SIZE = 640
PATIENCE = 20

print(f"\nEpochs: {EPOCHS} | Batch: {BATCH} | Size: {IMG_SIZE}")

In [ ]:
print("\n" + "="*60)
print("DATASET ANALYSIS")
print("="*60)

# Find images
train_imgs = list(Path(DATASET).rglob('**/train/**/*.jpg')) + \
             list(Path(DATASET).rglob('**/train/**/*.png'))
val_imgs = list(Path(DATASET).rglob('**/val*/**/*.jpg')) + \
           list(Path(DATASET).rglob('**/val*/**/*.png'))

print(f"Train: {len(train_imgs)} images")
print(f"Val: {len(val_imgs)} images")

# Find or create data.yaml
yaml_files = list(Path(DATASET).rglob('*.yaml'))
if yaml_files:
    data_yaml = str(yaml_files[0])
    with open(data_yaml) as f:
        config = yaml.safe_load(f)
        classes = config.get('names', [])
    print(f"✓ Found data.yaml at: {data_yaml}")
else:
    # Create data.yaml for license plate detection
    classes = ['license_plate']
    
    # Detect dataset structure
    train_path = 'train/images'
    val_path = 'valid/images'
    
    # Check alternative structures
    if not os.path.exists(f"{DATASET}/train/images"):
        if os.path.exists(f"{DATASET}/train"):
            train_path = 'train'
        elif os.path.exists(f"{DATASET}/images/train"):
            train_path = 'images/train'
    
    if not os.path.exists(f"{DATASET}/valid/images"):
        if os.path.exists(f"{DATASET}/valid"):
            val_path = 'valid'
        elif os.path.exists(f"{DATASET}/val/images"):
            val_path = 'val/images'
        elif os.path.exists(f"{DATASET}/val"):
            val_path = 'val'
        elif os.path.exists(f"{DATASET}/images/val"):
            val_path = 'images/val'
    
    config = {
        'path': DATASET,
        'train': train_path,
        'val': val_path,
        'nc': len(classes),
        'names': classes
    }
    data_yaml = f"{OUTPUT_DIR}/data.yaml"
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    with open(data_yaml, 'w') as f:
        yaml.dump(config, f)
    print(f"✓ Created data.yaml at: {data_yaml}")

print(f"Classes: {classes}")

# Show dataset structure
print("\n📂 Dataset structure:")
for root, dirs, files in os.walk(DATASET):
    level = root.replace(DATASET, '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}📁 {os.path.basename(root)}/')
    if level < 2:
        subindent = '  ' * (level + 1)
        for d in dirs[:5]:
            print(f'{subindent}📁 {d}/')
        if len(dirs) > 5:
            print(f'{subindent}... and {len(dirs)-5} more folders')

In [ ]:
print("\n" + "🎯"*30)
print("TRAINING LICENSE PLATE DETECTION MODEL")
print("🎯"*30)

try:
    model = YOLO('yolov8n.pt')
    
    print(f"\n🚀 Starting training...")
    print(f"   Output will be saved to: {MODEL_DIR}")
    print("="*60)
    
    results = model.train(
        data=data_yaml,
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH,
        project=OUTPUT_DIR,
        name='license_plate_training',
        patience=PATIENCE,
        save=True,
        device=0,
        plots=True,
        exist_ok=True,
        
        # Fixes for common errors
        amp=False,
        workers=0,
        cache=False,
    )
    
    # Copy best model to guaranteed location
    trained_model = f"{OUTPUT_DIR}/license_plate_training/weights/best.pt"
    if os.path.exists(trained_model):
        shutil.copy2(trained_model, f"{MODEL_DIR}/best.pt")
        shutil.copy2(f"{OUTPUT_DIR}/license_plate_training/weights/last.pt", f"{MODEL_DIR}/last.pt")
        print("\n" + "="*60)
        print("✅ TRAINING COMPLETE!")
        print("="*60)
        print(f"✓ Model saved to: {MODEL_DIR}/best.pt")
    else:
        print("\n⚠️  Warning: Model not found at expected location")
    
except Exception as e:
    print(f"\n❌ ERROR: {e}")

In [ ]:
print("\n" + "="*60)
print("VALIDATION")
print("="*60)

model_path = f"{MODEL_DIR}/best.pt"

if os.path.exists(model_path):
    model = YOLO(model_path)
    metrics = model.val(data=data_yaml)
    
    print(f"\n📊 RESULTS:")
    print(f"   mAP50-95: {metrics.box.map:.4f}")
    print(f"   mAP50:    {metrics.box.map50:.4f}")
    print(f"   Precision: {metrics.box.mp:.4f}")
    print(f"   Recall:    {metrics.box.mr:.4f}")
    
    # Show plots
    results_plot = f"{OUTPUT_DIR}/license_plate_training/results.png"
    if os.path.exists(results_plot):
        shutil.copy2(results_plot, f"{OUTPUT_DIR}/results.png")
        print("\n📈 Training curves:")
        display(IPImage(filename=results_plot))
    
    confusion_plot = f"{OUTPUT_DIR}/license_plate_training/confusion_matrix.png"
    if os.path.exists(confusion_plot):
        shutil.copy2(confusion_plot, f"{OUTPUT_DIR}/confusion_matrix.png")
        print("\n📈 Confusion matrix:")
        display(IPImage(filename=confusion_plot))
else:
    print(f"❌ Model not found at {model_path}")

In [ ]:
print("\n" + "="*60)
print("TESTING")
print("="*60)

if val_imgs and os.path.exists(model_path):
    samples = np.random.choice(val_imgs, min(6, len(val_imgs)), replace=False)
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    axes = axes.ravel()
    
    for i, img_path in enumerate(samples):
        results = model(str(img_path), conf=0.25)
        annotated = results[0].plot()
        annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
        
        axes[i].imshow(annotated_rgb)
        axes[i].set_title(f"Test {i+1}", fontsize=10)
        axes[i].axis('off')
    
    plt.suptitle("License Plate Detection Results", fontsize=14, weight='bold')
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/predictions.png', dpi=100)
    plt.show()
    print(f"✓ Saved predictions to: {OUTPUT_DIR}/predictions.png")

In [ ]:
print("\n" + "="*60)
print("EXPORT")
print("="*60)

if os.path.exists(model_path):
    try:
        model = YOLO(model_path)
        
        onnx_path = model.export(format='onnx')
        if os.path.exists(onnx_path):
            shutil.copy2(onnx_path, f"{MODEL_DIR}/license_plate_model.onnx")
            print(f"✓ ONNX: {MODEL_DIR}/license_plate_model.onnx")
        
        print("✓ Export complete")
    except Exception as e:
        print(f"⚠️  Export warning: {e}")

In [ ]:
print("\n" + "="*60)
print("🎉 COMPLETE - OUTPUT VERIFICATION")
print("="*60)

# Verify all outputs
outputs = {
    'Model (best.pt)': f"{MODEL_DIR}/best.pt",
    'Model (last.pt)': f"{MODEL_DIR}/last.pt",
    'ONNX Export': f"{MODEL_DIR}/license_plate_model.onnx",
    'Training Curves': f"{OUTPUT_DIR}/results.png",
    'Confusion Matrix': f"{OUTPUT_DIR}/confusion_matrix.png",
    'Predictions': f"{OUTPUT_DIR}/predictions.png",
}

print("\n📍 OUTPUT FILES:")
all_exist = True
for name, path in outputs.items():
    if os.path.exists(path):
        size = os.path.getsize(path) / (1024 * 1024)
        print(f"   ✅ {name}")
        print(f"      {path} ({size:.2f} MB)")
    else:
        print(f"   ❌ {name} - NOT FOUND")
        all_exist = False

# Create download package
print("\n📦 Creating download package...")
zip_path = '/kaggle/working/license_plate_detection_model'
shutil.make_archive(zip_path, 'zip', OUTPUT_DIR)
print(f"✓ Created: {zip_path}.zip")

# Summary
print(f"""
{'='*60}
LICENSE PLATE DETECTION - TRAINING SUMMARY
{'='*60}

Dataset: {len(train_imgs)} train, {len(val_imgs)} val images
Model: YOLOv8-Nano
Epochs: {EPOCHS}
Classes: {classes}

Files Location: {OUTPUT_DIR}
Main Model: {MODEL_DIR}/best.pt
Download Package: {zip_path}.zip

📥 HOW TO DOWNLOAD:
{'='*60}
METHOD 1: Individual files
   1. Left sidebar → 📁 folder icon
   2. Navigate to: working/outputs/model/
   3. Right-click best.pt → Download

METHOD 2: Complete package
   1. Right sidebar → Output tab
   2. Download: license_plate_detection_model.zip
{'='*60}
""")

print("\n✨ LICENSE PLATE DETECTION MODEL READY!")
print("="*60)